# ReVoice — Voice Cloning API (XTTS-v2 + F5-TTS)

**Before starting:** Runtime → Change runtime type → **T4 GPU** → Save

Run cells **top to bottom, once each**. Don't run any old cells from previous notebooks.

Licence note: F5-TTS model weights are released for **non-commercial use** (fine for a college project).

## Step 1 — Check the GPU

In [ ]:
!nvidia-smi

## Step 2 — Install everything (3–5 minutes)
Installs XTTS, then F5-TTS **without letting it change other packages**, then pins the versions that work.
**After this finishes: Runtime → Restart session**, then continue from Step 2b.

In [ ]:
# XTTS + server tools
!pip install -q coqui-tts fastapi uvicorn pyngrok soundfile

# F5-TTS: --no-deps so it can't overwrite transformers/numpy
!pip install -q --no-deps f5-tts cached_path
!pip install -q ema_pytorch torchdiffeq vocos x_transformers hydra-core omegaconf jieba pypinyin unidecode pydub wandb
!pip install -q rjieba || echo "rjieba optional - skipped"

# pin the versions XTTS needs (must come LAST so nothing overrides them)
!pip install -q --no-deps --force-reinstall "transformers>=4.57,<5"
!pip install -q "numpy<2.3"

!apt-get -qq install -y ffmpeg > /dev/null
print("Install finished. Now: Runtime -> Restart session, then run Step 2b.")

## Step 2b — Check versions (after restarting)
If this errors, don't continue.

In [ ]:
import transformers, numpy, librosa
from transformers.pytorch_utils import isin_mps_friendly
from transformers.utils.import_utils import is_torch_greater_or_equal
print("transformers", transformers.__version__)
print("numpy", numpy.__version__)
import f5_tts.api
print("f5-tts importable")
print("All checks passed")

## Step 3 — Load XTTS-v2 (~2 GB download first time)

In [ ]:
import os
os.environ["COQUI_TOS_AGREED"] = "1"

import torch
from TTS.api import TTS

device = "cuda" if torch.cuda.is_available() else "cpu"
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
xtts_model = tts.synthesizer.tts_model
SAMPLE_RATE = 24000
print("XTTS-v2 loaded on", device)

## Step 3b — Load F5-TTS (~1.5 GB download first time)
If this fails, XTTS still works — the app will report the F5 error when you pick it.

In [ ]:
f5 = None
try:
    from f5_tts.api import F5TTS
    f5 = F5TTS(device=device)
    print("F5-TTS loaded")
except Exception as e:
    import traceback; traceback.print_exc()
    print("\nF5-TTS failed to load (XTTS still available):", e)

## Step 4 — Quick test of both models

In [ ]:
from IPython.display import Audio, display
from importlib.resources import files

print("XTTS-v2:")
display(Audio(tts.tts(text="Hello, this is the XTTS voice.", speaker="Ana Florence", language="en"), rate=SAMPLE_RATE))

if f5 is not None:
    print("F5-TTS:")
    ref = str(files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
    wav, sr, _ = f5.infer(ref_file=ref,
                          ref_text="Some call me nature, others call me mother nature.",
                          gen_text="Hello, this is the F5 voice.")
    display(Audio(wav, rate=sr))

## Step 5 — ngrok token
https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
NGROK_AUTH_TOKEN = "PASTE_YOUR_NGROK_TOKEN_HERE"

## Step 6 — Start the API (both engines)
Safe to re-run any time. Your ngrok URL stays the same.
The last line must say **ReVoice API v5**.
v5 adds: stock (built-in) XTTS voices, `GET /speakers`, and `POST /forget_voices`
(the backend calls it when a user is deleted, so their voice is wiped here too).

In [ ]:
import os, traceback, threading, time, requests, hashlib, socket
from fastapi import FastAPI
from fastapi.responses import JSONResponse
from pydantic import BaseModel
from typing import List, Optional
import base64, io, tempfile, subprocess
import soundfile as sf
import uvicorn
from pyngrok import ngrok

try:
    server.should_exit = True; time.sleep(2)
except NameError:
    pass
try:
    ngrok.kill()
except Exception:
    pass

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

xtts_cache = {}   # voice profiles for XTTS
f5_cache = {}     # (reference clip, transcript) for F5
temp_files = []   # cleaned clips on disk, deleted by /forget_voices

EMOTION_STYLE = {
    "neutral": {"speed": 1.00, "temperature": 0.80},
    "happy":   {"speed": 1.05, "temperature": 0.85},
    "urgent":  {"speed": 1.12, "temperature": 0.85},
    "tender":  {"speed": 0.92, "temperature": 0.75},
}

app = FastAPI()

class SynthRequest(BaseModel):
    text: str
    speaker_wav_base64: Optional[str] = None
    speaker_wav_base64_list: Optional[List[str]] = None
    language: str = "en"
    emotion: str = "neutral"
    engine: str = "xtts"
    stock_speaker: Optional[str] = None   # built-in XTTS voice, for users with no recordings

def clean_clip(b64, max_seconds=None):
    """Decode a browser recording and clean it for cloning."""
    raw = base64.b64decode(b64)
    f = tempfile.NamedTemporaryFile(suffix=".input", delete=False)
    f.write(raw); f.close()
    out = f.name + ".wav"
    cmd = ["ffmpeg", "-y", "-i", f.name, "-ac", "1", "-ar", "24000",
           "-af", "highpass=f=70,"
                  "silenceremove=start_periods=1:start_silence=0.1:start_threshold=-45dB,"
                  "loudnorm=I=-18:TP=-2:LRA=11"]
    if max_seconds:
        cmd += ["-t", str(max_seconds)]
    r = subprocess.run(cmd + [out], capture_output=True, text=True)
    os.remove(f.name)              # the raw upload isn't needed any more
    if r.returncode != 0:
        raise RuntimeError("ffmpeg failed:\n" + r.stderr[-1200:])
    temp_files.append(out)
    return out

def usable_clips(clips_b64, max_seconds=None):
    out = []
    for b64 in clips_b64:
        p = clean_clip(b64, max_seconds)
        d = sf.info(p).duration
        if d >= 0.8:
            out.append((p, d))
    if not out:
        raise RuntimeError("All reference clips were too short or silent. Re-record and keep talking.")
    return out

# ---------- XTTS ----------
def run_xtts(req, style):
    key = hashlib.md5("".join(req.speaker_wav_base64_list).encode()).hexdigest()
    if key not in xtts_cache:
        clips = usable_clips(req.speaker_wav_base64_list)
        print(f"   XTTS profile from {sum(d for _, d in clips):.1f}s of audio")
        xtts_cache[key] = xtts_model.get_conditioning_latents(
            audio_path=[p for p, _ in clips], gpt_cond_len=30, gpt_cond_chunk_len=4, max_ref_length=30)
    gpt_cond_latent, speaker_embedding = xtts_cache[key]
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE

# ---------- XTTS with a stock (built-in) voice ----------
def stock_speaker_names():
    return sorted(xtts_model.speaker_manager.speaker_names)

def run_xtts_stock(req, style):
    speakers = xtts_model.speaker_manager.speakers
    if req.stock_speaker not in speakers:
        raise RuntimeError(f"Unknown stock voice {req.stock_speaker!r}. "
                           f"Some valid ones: {', '.join(stock_speaker_names()[:8])}")
    # built-in voices come with their voice profile already computed
    gpt_cond_latent, speaker_embedding = speakers[req.stock_speaker].values()
    out = xtts_model.inference(
        req.text, req.language, gpt_cond_latent, speaker_embedding,
        temperature=style["temperature"], speed=style["speed"],
        length_penalty=1.0, repetition_penalty=10.0, top_k=50, top_p=0.85,
        enable_text_splitting=True)
    return out["wav"], SAMPLE_RATE

# ---------- F5-TTS ----------
def run_f5(req, style):
    if f5 is None:
        raise RuntimeError("F5-TTS isn't loaded. Check the output of Step 3b.")
    key = hashlib.md5("".join(req.speaker_wav_base64_list).encode()).hexdigest()
    if key not in f5_cache:
        # F5 works best with ONE clean reference of under ~12s, so use the longest clip, trimmed
        clips = usable_clips(req.speaker_wav_base64_list, max_seconds=11)
        ref_path = max(clips, key=lambda c: c[1])[0]
        try:
            ref_text = f5.transcribe(ref_path)        # Whisper writes down what you said
        except Exception as e:
            print("   transcription failed, letting F5 handle it:", e)
            ref_text = ""
        print(f"   F5 reference: {sf.info(ref_path).duration:.1f}s, transcript: {ref_text!r}")
        f5_cache[key] = (ref_path, ref_text)
    ref_path, ref_text = f5_cache[key]
    wav, sr, _ = f5.infer(ref_file=ref_path, ref_text=ref_text, gen_text=req.text,
                          speed=style["speed"], nfe_step=32, show_info=lambda *a, **k: None)
    return wav, sr

@app.get("/")
def health():
    return {"status": "ReVoice API v5 running", "xtts": True, "f5": f5 is not None}

@app.get("/speakers")
def speakers():
    """Names of XTTS's built-in voices (for the 'use a stock voice' option)."""
    return {"speakers": stock_speaker_names()}

@app.post("/forget_voices")
def forget_voices():
    """Wipes every cached voice profile and cleaned clip (called when a user is deleted)."""
    n = len(xtts_cache) + len(f5_cache)
    xtts_cache.clear(); f5_cache.clear()
    for path in temp_files:
        try:
            os.remove(path)
        except OSError:
            pass
    temp_files.clear()
    print(f"\n--- forgot {n} cached voice(s)")
    return {"forgotten": n}

@app.post("/synthesize")
def synthesize(req: SynthRequest):
    try:
        style = EMOTION_STYLE.get(req.emotion, EMOTION_STYLE["neutral"])
        t0 = time.time()

        if req.stock_speaker:
            # stock voice: no recordings needed (XTTS only)
            if req.engine != "xtts":
                raise RuntimeError("Stock voices only work with XTTS-v2.")
            print(f"\n--- [xtts stock] {req.text!r} | {req.emotion} | {req.stock_speaker}")
            wav, sr = run_xtts_stock(req, style)
        else:
            if not req.speaker_wav_base64_list and req.speaker_wav_base64:
                req.speaker_wav_base64_list = [req.speaker_wav_base64]
            if not req.speaker_wav_base64_list:
                raise RuntimeError("No reference audio sent.")
            print(f"\n--- [{req.engine}] {req.text!r} | {req.emotion} | {len(req.speaker_wav_base64_list)} clip(s)")
            wav, sr = run_f5(req, style) if req.engine == "f5" else run_xtts(req, style)

        print(f"   generated in {time.time() - t0:.1f}s")
        buf = io.BytesIO()
        sf.write(buf, wav, samplerate=sr, format="WAV")
        return {"audio_base64": base64.b64encode(buf.getvalue()).decode("utf-8")}
    except Exception as e:
        print(traceback.format_exc())
        return JSONResponse({"error": f"{type(e).__name__}: {e}"}, status_code=500)

def free_port():
    s = socket.socket(); s.bind(("", 0)); p = s.getsockname()[1]; s.close(); return p

PORT = free_port()
server = uvicorn.Server(uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning"))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)

tunnel = ngrok.connect(PORT)
print("=" * 65)
print("URL for COLAB_API_URL in backend/app.py:")
print(tunnel.public_url)
print("=" * 65)
print(requests.get(tunnel.public_url, headers={"ngrok-skip-browser-warning": "true"}).json())

## Keep this tab open
If Colab disconnects: run Steps 3, 3b and 6 again (Step 2 only if packages are missing).
Never run cells from older notebooks — the URL always points to whichever server started last.